# Notebook 06 — Centroid C: Trajectory-Averaged Centroid

For each conversation, average `h_inst` across **all turns** to get a single
trajectory-averaged vector. Build centroids from these, then evaluate against:
1. Mode B test data — trajectory-averaged vectors (attempt 5)
2. Mode A test data — final-turn vectors (attempt 10)

**Key question:** does trajectory averaging give better/worse separation than final-turn only?

## Section 1: Setup & Load

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

MODEB_ROOT   = repo_root / "results" / "representations" / "mode_b"
MODEA_ROOT   = repo_root / "results" / "representations" / "mode_a"
CENTROID_DIR = repo_root / "results" / "centroids"
FIG_DIR      = repo_root / "notebooks" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

FRAMEWORKS = ["crescendo", "actorattack", "xteaming"]
FW_PALETTE = {"crescendo": "#4C72B0", "actorattack": "#DD8452", "xteaming": "#55A868"}
N_LAYERS   = 32
HIDDEN_DIM = 4096

print("repo_root:", repo_root)

In [ ]:
# Load Mode B arrays (memory-mapped)
modeb = {}  # key -> {"meta": DataFrame, "h_inst": mmap}

for fw in FRAMEWORKS:
    for gt in ("harmful", "benign"):
        key    = f"{fw}_{gt}"
        folder = MODEB_ROOT / key
        if not folder.exists():
            print(f"  MISSING: {key}")
            continue
        meta   = pd.read_parquet(folder / "metadata.parquet")
        h_inst = np.load(str(folder / "h_inst.npy"), mmap_mode="r")
        modeb[key] = {"meta": meta, "h_inst": h_inst}
        print(f"  {key}: {meta.shape[0]} rows, h_inst {h_inst.shape}")

# Build combined per-framework DataFrames
fw_dfs = {}
for fw in FRAMEWORKS:
    parts = []
    for gt in ("harmful", "benign"):
        key = f"{fw}_{gt}"
        if key not in modeb:
            continue
        m = modeb[key]["meta"].copy()
        m["_key"] = key
        m["_idx"] = np.arange(len(m))
        parts.append(m)
    fw_dfs[fw] = pd.concat(parts, ignore_index=True)

print("\nCombined sizes:", {fw: len(df) for fw, df in fw_dfs.items()})

## Section 2: Trajectory Averaging

For each `conversation_id`, average `h_inst` across all turns:
```
h_avg = mean over turn_k of h_inst[k]   # (32, 4096)
```
This collapses the turn dimension to one vector per conversation.

In [ ]:
def get_h(rows_df, store):
    """Fetch h_inst rows from mmap store; returns float32 (N, 32, 4096)."""
    parts = []
    for key, grp in rows_df.groupby("_key"):
        arr   = store[key]["h_inst"]
        chunk = arr[grp["_idx"].values].astype(np.float32)
        parts.append(chunk)
    return np.concatenate(parts, axis=0) if parts else np.empty((0, N_LAYERS, HIDDEN_DIM), np.float32)


def trajectory_average(df, store):
    """
    For each conversation_id in df, average h_inst over all its turns.
    Returns a new DataFrame (one row per conversation) with an 'h_avg' column
    holding (32, 4096) arrays, plus metadata from the final turn.
    """
    conv_rows = []
    for conv_id, grp in df.groupby("conversation_id"):
        h = get_h(grp, store)   # (n_turns, 32, 4096)
        h_avg = h.mean(axis=0)  # (32, 4096)
        # Keep metadata from the last turn of the conversation
        meta_row = grp.sort_values("turn_k").iloc[-1].to_dict()
        meta_row["h_avg"] = h_avg
        conv_rows.append(meta_row)
    return pd.DataFrame(conv_rows)


# Build trajectory-averaged DataFrames for train and test splits
fw_traj_train = {}  # fw -> averaged train DataFrame
fw_traj_test  = {}  # fw -> averaged test DataFrame

for fw in FRAMEWORKS:
    df    = fw_dfs[fw]
    train = df[df["attempt"] < 5].reset_index(drop=True)
    test  = df[df["attempt"] == 5].reset_index(drop=True)
    fw_traj_train[fw] = trajectory_average(train, modeb)
    fw_traj_test[fw]  = trajectory_average(test,  modeb)
    print(f"  {fw}: train={len(fw_traj_train[fw])} test={len(fw_traj_test[fw])} conversations")

## Section 3: Centroid C Construction

In [ ]:
def build_centroid_C(train_traj_df):
    """
    train_traj_df: one row per conversation with 'h_avg' (32,4096) array.
    Returns dict: v_harmful, mu_harmful, mu_harmless.
    """
    harm = train_traj_df[train_traj_df["goal_type"] == "harmful"]
    ben  = train_traj_df[train_traj_df["goal_type"] == "benign"]

    accepted_ben = ben[ben["final_turn_outcome"] == "accepted"] if "final_turn_outcome" in ben.columns else ben

    def stack_h_avg(sub):
        if len(sub) == 0:
            return np.zeros((0, N_LAYERS, HIDDEN_DIM), np.float32)
        return np.stack(sub["h_avg"].values, axis=0).astype(np.float32)

    mu_harmful  = stack_h_avg(harm).mean(axis=0)         if len(harm) > 0 else np.zeros((N_LAYERS, HIDDEN_DIM), np.float32)
    mu_harmless = stack_h_avg(accepted_ben).mean(axis=0) if len(accepted_ben) > 0 else np.zeros((N_LAYERS, HIDDEN_DIM), np.float32)

    # v_harmful: within-pair subtraction
    v = np.zeros((N_LAYERS, HIDDEN_DIM), dtype=np.float32)
    n_pairs = 0
    for pair_id, grp in train_traj_df.groupby("pair_id"):
        h_grp = grp[grp["goal_type"] == "harmful"]
        b_grp = grp[grp["goal_type"] == "benign"]
        if len(h_grp) == 0 or len(b_grp) == 0:
            continue
        h_vec = stack_h_avg(h_grp).mean(axis=0)  # (32, 4096)
        b_vec = stack_h_avg(b_grp).mean(axis=0)
        delta = h_vec - b_vec
        norm  = np.linalg.norm(delta, axis=-1, keepdims=True).clip(min=1e-8)
        v    += delta / norm
        n_pairs += 1
    if n_pairs > 0:
        v /= n_pairs

    return {"v_harmful": v, "mu_harmful": mu_harmful, "mu_harmless": mu_harmless}


centroids_C = {}  # fw -> centroid dict
for fw in FRAMEWORKS:
    centroids_C[fw] = build_centroid_C(fw_traj_train[fw])
    print(f"  Built Centroid C: {fw}")

## Section 4: Evaluation Helpers

In [ ]:
def delta_score_batch(h_arr, mu_harmful, mu_harmless):
    """h_arr: (N,32,4096) float32. Returns (N,) scores."""
    h  = torch.tensor(h_arr,       dtype=torch.float32)
    mh = torch.tensor(mu_harmful,  dtype=torch.float32)
    mb = torch.tensor(mu_harmless, dtype=torch.float32)
    h_n  = F.normalize(h,  dim=2)
    mh_n = F.normalize(mh, dim=1)
    mb_n = F.normalize(mb, dim=1)
    sim_h = (h_n * mh_n.unsqueeze(0)).sum(dim=2)
    sim_b = (h_n * mb_n.unsqueeze(0)).sum(dim=2)
    return (sim_h - sim_b).mean(dim=1).numpy()


def eval_accuracy(h_arr, goal_types, centroid):
    """Compute harm_acc, ben_acc, overall_acc, n_harm, n_ben."""
    scores = delta_score_batch(h_arr, centroid["mu_harmful"], centroid["mu_harmless"])
    is_harm = np.array(goal_types) == "harmful"
    is_ben  = ~is_harm
    harm_acc = (scores[is_harm] > 0).mean() if is_harm.any() else np.nan
    ben_acc  = (scores[is_ben] <= 0).mean() if is_ben.any() else np.nan
    n_h, n_b = is_harm.sum(), is_ben.sum()
    overall  = (harm_acc * n_h + ben_acc * n_b) / (n_h + n_b) if (n_h + n_b) > 0 else np.nan
    return {"harm_acc": harm_acc, "ben_acc": ben_acc, "overall_acc": overall,
            "n_harm": int(n_h), "n_ben": int(n_b)}


def plot_eval_matrix(records_df, title, fname_stem, sources, test_fws=None):
    if test_fws is None:
        test_fws = FRAMEWORKS
    fig, axes = plt.subplots(1, 3, figsize=(5 * 3, 4))
    fig.suptitle(title, fontsize=13)

    for ax, (metric, label, cmap) in zip(axes, [
        ("harm_acc",    "Harmful acc",  "Reds"),
        ("ben_acc",     "Benign acc",   "Greens"),
        ("overall_acc", "Overall acc",  "Blues"),
    ]):
        mat   = records_df.pivot(index="centroid_source", columns="test_framework", values=metric)\
                          .reindex(index=sources, columns=test_fws)
        n_col = records_df.pivot(index="centroid_source", columns="test_framework", values="n_harm")\
                          .reindex(index=sources, columns=test_fws)
        annot = mat.applymap(lambda x: f"{x:.2f}" if not np.isnan(x) else "n/a") + \
                "\n" + n_col.applymap(lambda x: f"n={int(x)}" if not np.isnan(x) else "")
        sns.heatmap(mat.astype(float), annot=annot.values, fmt="", ax=ax,
                    vmin=0, vmax=1, cmap=cmap, linewidths=0.5)
        ax.set_title(label)
        ax.set_xlabel("Test framework")
        ax.set_ylabel("Centroid source")

    plt.tight_layout()
    path = FIG_DIR / f"{fname_stem}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {path}")

## Section 5: Eval 1 — Mode B Test (Trajectory-Averaged)

In [ ]:
records_modeb = []

for cent_fw in FRAMEWORKS:
    centroid = centroids_C[cent_fw]
    for test_fw in FRAMEWORKS:
        test_df = fw_traj_test[test_fw]
        if len(test_df) == 0:
            continue
        h_arr      = np.stack(test_df["h_avg"].values, axis=0).astype(np.float32)
        goal_types = test_df["goal_type"].values
        r = eval_accuracy(h_arr, goal_types, centroid)
        records_modeb.append({"centroid_source": cent_fw, "test_framework": test_fw, **r})

modeb_eval_df = pd.DataFrame(records_modeb)

plot_eval_matrix(
    modeb_eval_df,
    title="Centroid C — Mode B test (trajectory-averaged)",
    fname_stem="06_centroid_C_modeb",
    sources=FRAMEWORKS,
)

## Section 6: Load Mode A Test Data (Final-Turn, Attempt 10)

In [ ]:
modea = {}  # key -> {"meta": DataFrame, "h_inst": mmap}

for fw in FRAMEWORKS:
    for gt in ("harmful", "benign"):
        key    = f"{fw}_{gt}"
        folder = MODEA_ROOT / key
        if not folder.exists():
            print(f"  MISSING mode_a: {key}")
            continue
        meta   = pd.read_parquet(folder / "metadata.parquet")
        h_inst = np.load(str(folder / "h_inst.npy"), mmap_mode="r")
        # Mode A: attempt==10 is the test split
        test_mask = meta["attempt"] == 10 if "attempt" in meta.columns else pd.Series([True] * len(meta))
        meta_test = meta[test_mask].reset_index(drop=True)
        modea[key] = {"meta": meta_test, "h_inst": h_inst, "test_mask_idx": meta[test_mask].index.values}
        print(f"  {key}: {len(meta_test)} test rows")

# Combine into per-framework test DataFrames
fw_modea_test = {}
for fw in FRAMEWORKS:
    parts = []
    for gt in ("harmful", "benign"):
        key = f"{fw}_{gt}"
        if key not in modea:
            continue
        m = modea[key]["meta"].copy()
        m["_key"]  = key
        m["_orig_idx"] = modea[key]["test_mask_idx"]
        parts.append(m)
    if parts:
        fw_modea_test[fw] = pd.concat(parts, ignore_index=True)

print("\nMode A test sizes:", {fw: len(df) for fw, df in fw_modea_test.items()})

## Section 7: Eval 2 — Mode A Test (Final-Turn) Using Trajectory Centroid

In [ ]:
def get_h_modea(rows_df):
    """Fetch h_inst for mode_a test rows; returns float32 (N, 32, 4096)."""
    parts = []
    for key, grp in rows_df.groupby("_key"):
        arr   = modea[key]["h_inst"]
        chunk = arr[grp["_orig_idx"].values].astype(np.float32)
        parts.append(chunk)
    return np.concatenate(parts, axis=0) if parts else np.empty((0, N_LAYERS, HIDDEN_DIM), np.float32)


records_modea = []

for cent_fw in FRAMEWORKS:
    centroid = centroids_C[cent_fw]
    for test_fw in FRAMEWORKS:
        if test_fw not in fw_modea_test:
            continue
        test_df = fw_modea_test[test_fw]
        h_arr   = get_h_modea(test_df)
        if len(h_arr) == 0:
            continue
        goal_types = test_df["goal_type"].values
        r = eval_accuracy(h_arr, goal_types, centroid)
        records_modea.append({"centroid_source": cent_fw, "test_framework": test_fw, **r})

modea_eval_df = pd.DataFrame(records_modea)

plot_eval_matrix(
    modea_eval_df,
    title="Centroid C — Mode A test (final-turn, attempt 10)",
    fname_stem="06_centroid_C_modea",
    sources=FRAMEWORKS,
)

## Section 8: Comparison vs Notebook 04 Final-Turn Results

In [ ]:
# Load notebook 04 centroids for comparison (same-source diagonal)
nb04_records = []

for fw in FRAMEWORKS:
    cent_path = CENTROID_DIR / f"{fw}_centroids.pt"
    if not cent_path.exists():
        print(f"  MISSING: {cent_path}")
        continue
    c = torch.load(cent_path, weights_only=False)
    mu_h = c["mu_harmful"].numpy()  if isinstance(c["mu_harmful"],  torch.Tensor) else c["mu_harmful"]
    mu_b = c["mu_harmless"].numpy() if isinstance(c["mu_harmless"], torch.Tensor) else c["mu_harmless"]
    centroid_nb04 = {"mu_harmful": mu_h, "mu_harmless": mu_b}

    for test_fw in FRAMEWORKS:
        if test_fw not in fw_modea_test:
            continue
        test_df = fw_modea_test[test_fw]
        h_arr   = get_h_modea(test_df)
        if len(h_arr) == 0:
            continue
        goal_types = test_df["goal_type"].values
        r = eval_accuracy(h_arr, goal_types, centroid_nb04)
        nb04_records.append({"centroid_source": f"{fw}_nb04", "test_framework": test_fw, **r})

nb04_eval_df = pd.DataFrame(nb04_records)

# Side-by-side summary table
print("Overall accuracy — Notebook 04 final-turn centroid vs Centroid C (trajectory-averaged)")
print("Evaluated on Mode A test (attempt=10)\n")

rows_all = []
for cent_fw in FRAMEWORKS:
    for test_fw in FRAMEWORKS:
        c4 = nb04_eval_df[
            (nb04_eval_df["centroid_source"] == f"{cent_fw}_nb04") &
            (nb04_eval_df["test_framework"]  == test_fw)
        ]["overall_acc"]
        cC = modea_eval_df[
            (modea_eval_df["centroid_source"] == cent_fw) &
            (modea_eval_df["test_framework"]  == test_fw)
        ]["overall_acc"]
        acc4 = c4.iloc[0] if len(c4) > 0 else np.nan
        accC = cC.iloc[0] if len(cC) > 0 else np.nan
        rows_all.append({
            "centroid_fw": cent_fw, "test_fw": test_fw,
            "nb04_final_turn": acc4, "centroid_C_traj": accC,
            "delta": accC - acc4 if not (np.isnan(acc4) or np.isnan(accC)) else np.nan
        })

compare_df = pd.DataFrame(rows_all)
print(compare_df.round(3).to_string(index=False))

In [ ]:
# Visualise comparison: grouped bars per test framework
fig, axes = plt.subplots(1, len(FRAMEWORKS), figsize=(5 * len(FRAMEWORKS), 4), sharey=True)
fig.suptitle("Centroid C vs Notebook 04 — Overall accuracy on Mode A test", fontsize=13)

x = np.arange(len(FRAMEWORKS))
width = 0.35

for ax, test_fw in zip(axes, FRAMEWORKS):
    sub = compare_df[compare_df["test_fw"] == test_fw]
    nb04_vals = [sub[sub["centroid_fw"] == fw]["nb04_final_turn"].values[0]
                 if len(sub[sub["centroid_fw"] == fw]) > 0 else np.nan for fw in FRAMEWORKS]
    trajC_vals = [sub[sub["centroid_fw"] == fw]["centroid_C_traj"].values[0]
                  if len(sub[sub["centroid_fw"] == fw]) > 0 else np.nan for fw in FRAMEWORKS]

    bars1 = ax.bar(x - width/2, nb04_vals,  width, label="NB04 final-turn", alpha=0.8, color="steelblue")
    bars2 = ax.bar(x + width/2, trajC_vals, width, label="Centroid C traj", alpha=0.8, color="darkorange")
    ax.set_xticks(x); ax.set_xticklabels(FRAMEWORKS, rotation=20, ha="right")
    ax.set_title(f"Test: {test_fw}"); ax.set_ylabel("Overall accuracy")
    ax.set_ylim(0, 1.05); ax.axhline(0.5, linestyle="--", color="gray", linewidth=0.8, alpha=0.6)
    ax.legend(fontsize=8)

plt.tight_layout()
fname = FIG_DIR / "06_centroid_C_vs_nb04.png"
plt.savefig(fname, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {fname}")

## Section 9: Summary

In [ ]:
print("=" * 65)
print("SUMMARY: Centroid C — trajectory-averaged centroid")
print("=" * 65)

print("\nMode B test (trajectory-averaged) — diagonal (same-framework) overall accuracy:")
for fw in FRAMEWORKS:
    r = modeb_eval_df[(modeb_eval_df["centroid_source"] == fw) & (modeb_eval_df["test_framework"] == fw)]
    acc = r["overall_acc"].iloc[0] if len(r) > 0 else np.nan
    print(f"  {fw}: {acc:.3f}")

print("\nMode A test (final-turn, attempt 10) — diagonal overall accuracy:")
for fw in FRAMEWORKS:
    r = modea_eval_df[(modea_eval_df["centroid_source"] == fw) & (modea_eval_df["test_framework"] == fw)]
    acc = r["overall_acc"].iloc[0] if len(r) > 0 else np.nan
    print(f"  {fw}: {acc:.3f}")

print("\nMean delta (Centroid C - NB04 final-turn) across all cells:")
print(f"  {compare_df['delta'].mean():.3f}  (positive = trajectory avg better)")